# Reproducción del enriquecimiento de componentes ausentes

Misma metodología existente; no modifica recuperación ni publica resultados. Por defecto solo comprueba entradas públicas. Activar el cálculo consume API, crea resultados separados y exige revisión antes de publicarlos. No reutilizar checkpoints de código o prompts diferentes. No introducir claves en celdas ni outputs.


In [1]:
from pathlib import Path
import json, sys
candidates = [Path.cwd(), *Path.cwd().parents, Path(__file__).resolve().parent.parent] if '__file__' in globals() else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'src/risk/session_exposure_grounding.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Abra este notebook desde una copia del repositorio actualizado e instale requirements-test.txt.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repositorio localizado. No se solicitan credenciales.')


Repositorio localizado. No se solicitan credenciales.


In [2]:
import pandas as pd
from src.risk.enrich_missing_exposure import prepare_cases
signals=pd.read_csv(ROOT/'data/evaluation/risk_details/signals_published.csv')
chunks=pd.read_csv(ROOT/'data/processed/chunks/chunks_recursive.csv')
catalog=json.loads((ROOT/'config/project_catalog.json').read_text())
cases=prepare_cases(signals,chunks,catalog)
assert len(cases)==176
print(f'Casos pendientes con fuente y proyecto conciliados: {len(cases)}')


Casos pendientes con fuente y proyecto conciliados: 176


In [3]:
RUN_ENRICHMENT=False
if RUN_ENRICHMENT:
    import getpass, tempfile
    from openai import OpenAI
    from src.risk.enrich_missing_exposure import run
    baseline=json.loads((ROOT/'results/day_23/project_scope_context_reviewed.json').read_text())
    client=OpenAI(api_key=getpass.getpass('Clave API (entrada oculta): '))
    output=Path(tempfile.mkdtemp(prefix='exposure_review_'))
    summary=run(signals,chunks,catalog,baseline,client,output,batch_size=10)
    del client
    print('Resultado separado, pendiente de revisión:',output)
else:
    print('Verificación de entradas terminada. Sin llamadas API ni cambios al perfil publicado.')


Verificación de entradas terminada. Sin llamadas API ni cambios al perfil publicado.
